Deliverable_2

In [26]:

import re
import json
from pathlib import Path

import numpy as np
import pandas as pd

#Task 1 File ingestion and regular expressions
def read_file(path) -> tuple[str, str]:
    raw = Path(path).read_bytes()
    try:
        return raw.decode("utf-8"), "utf-8"
    except UnicodeDecodeError:
        return raw.decode("iso-8859-1"), "iso-8859-1"

#Compile regex pattern for log line parsing
LOG_PATTERN = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
    r"\[(?P<register_id>[^\]]+)\]\s+"
    r"item=(?P<item>.+?)\s+"
    r"price=(?P<price>\d+(?:\.\d+)?)\s+"
    r"(?P<currency>[A-Z]{3})\s+"
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$"
)

#Function to parse log lines into structured records
def parse_log(text: str) -> list[dict]:
    records = []
    for line in text.splitlines():
        match = LOG_PATTERN.match(line.strip().lstrip("\ufeff"))
        if match:
            record = match.groupdict()
            record["price"] = float(record["price"])
            records.append(record)
    return records

#Task 2

## Task 2: Recursive utility functions
### 2.1 Finding log files recursively



In [27]:
for item in Path("data").iterdir():
    print(item, "-> mappe" if item.is_dir() else "-> fil")

data/2026 -> mappe
data/store_config.json -> fil


In [28]:
def find_log_files(directory: str | Path, extension: str = ".log") -> list[Path]:
    """Finner alle filer med gitt filendelse rekursivt i en mappestruktur."""
    directory = Path(directory)
    log_files = []
    for item in directory.iterdir():
        if item.is_dir():
            log_files.extend(find_log_files(item, extension))  # funksjonen kaller seg selv
        elif item.is_file() and item.suffix == extension:
            log_files.append(item)
    return log_files

In [29]:
log_files = find_log_files("data")
print(f"Fant {len(log_files)} loggfiler")
for f in log_files[:5]:
    print(f)

Fant 9 loggfiler
data/2026/science_building/POS_01/register_sales.log
data/2026/science_building/POS_03/register_sales.log
data/2026/science_building/POS_02/register_sales.log
data/2026/student_union/POS_01/register_sales.log
data/2026/student_union/POS_03/register_sales.log


### 2.2 Active registers from store_config.json
The function traverses any combination of nested dicts and lists:
- **dict**: if it has `"id"` and `"active": true`, the id is added. Then we search all values.
- **list**: we search every element.
- **anything else** (string, number, bool): base case, nothing to do.

In [30]:
def find_active_registers(data) -> set[str]:
    """Traverserer nøstede dicts og lister og returnerer ID-ene til aktive kasser."""
    active = set()
    if isinstance(data, dict):
        if data.get("active") is True and "id" in data:
            active.add(data["id"])
        for value in data.values():
            active |= find_active_registers(value)   # rekursivt kall
    elif isinstance(data, list):
        for element in data:
            active |= find_active_registers(element)  # rekursivt kall
    return active

In [31]:
config = json.loads(Path("data/store_config.json").read_text(encoding="utf-8"))
active_registers = find_active_registers(config)
print(sorted(active_registers))

['POS_01', 'POS_02', 'POS_03']
